# ASL Fingerspelling Training Notebook

This notebook trains and tests every model variant described in the project context using the shared `engine/` training pipeline and the shared dataset loader in `engine/dataset.py`.

Models included:
- Custom CNN from scratch with SGD + momentum
- Custom CNN from scratch with Adam
- MobileNetV2 pretrained feature extraction
- MobileNetV2 from scratch
- InceptionV3 pretrained feature extraction
- InceptionV3 from scratch

The dataset loader creates separate train, validation, and test splits before training.

## Workflow Overview

This notebook prepares the Kaggle environment, detects the dataset root, loads data through `engine/dataset.py`, and then trains the full model matrix with a separate held-out test split.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import torch

def discover_repo_root():
    candidates = [Path('/kaggle/working'), Path.cwd().resolve()]
    candidates.extend(Path.cwd().resolve().parents)
    for candidate in candidates:
        if (candidate / 'requirements.txt').exists() and (candidate / 'engine').exists():
            return candidate
    for candidate in candidates:
        if (candidate / 'engine').exists():
            return candidate
    return Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd().resolve()

REPO_ROOT = discover_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def ensure_requirements():
    requirements_path = REPO_ROOT / 'requirements.txt'
    if requirements_path.exists():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])
        print(f'Installed dependencies from {requirements_path}')
    else:
        print('requirements.txt not found, skipping install.')

def find_dataset_root(input_root=Path('/kaggle/input')):
    if not input_root.exists():
        return None

    def looks_like_class_root(folder):
        return len([p for p in folder.iterdir() if p.is_dir()]) >= 2

    for candidate in sorted(p for p in input_root.iterdir() if p.is_dir()):
        if looks_like_class_root(candidate):
            return candidate
        for nested in sorted(p for p in candidate.rglob('*') if p.is_dir()):
            if looks_like_class_root(nested):
                return nested
    return None

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('Running on CPU.')

ensure_requirements()

from engine import config as cfg
from engine import train as trainer
from engine.dataset import get_data_loaders

dataset_root = find_dataset_root()
if dataset_root is None:
    dataset_root = REPO_ROOT / 'data' / 'processed'
    print(f'No Kaggle input dataset found. Falling back to {dataset_root}')
else:
    print(f'Dataset root detected: {dataset_root}')
    cfg.KAGGLE = True
    cfg.PROJECT_ROOT = REPO_ROOT

cfg.DATA_DIR = dataset_root

print('Configuration summary:')
print(json.dumps({
    'DATA_DIR': str(cfg.DATA_DIR),
    'MODEL_TYPE': cfg.MODEL_TYPE,
    'NUM_CLASSES': cfg.NUM_CLASSES,
    'BATCH_SIZE': cfg.BATCH_SIZE,
    'TRAIN_SPLIT': cfg.TRAIN_SPLIT,
    'VALIDATION_SPLIT': cfg.VALIDATION_SPLIT,
    'TEST_SPLIT': cfg.TEST_SPLIT,
}, indent=2))

Running on CPU.


## Environment Setup

In [ ]:
train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = get_data_loaders(
    data_dir=cfg.DATA_DIR,
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS
)

cfg.NUM_CLASSES = len(train_loader.dataset.subset.dataset.classes)

print('Split sizes:')
print(f'  Train: {len(train_dataset)}')
print(f'  Validation: {len(val_dataset)}')
print(f'  Test: {len(test_dataset)}')
print(f'  Classes: {cfg.NUM_CLASSES}')

## Training Plan

The next cell trains all requested configurations. Each run evaluates the best checkpoint on the held-out test split and stores a history JSON file in the notebook output folder.

The epoch counts are intentionally modest so the notebook remains runnable on Kaggle. Increase them if you want longer training runs.

In [ ]:
experiments = [
    {
        'name': 'cnn_sgd_momentum',
        'model_type': 'custom_cnn',
        'pretrained': False,
        'optimizer_name': 'sgd',
        'learning_rate': 1e-3,
        'momentum': 0.9,
        'weight_decay': 5e-4,
        'epochs': 5,
    },
    {
        'name': 'cnn_adam',
        'model_type': 'custom_cnn',
        'pretrained': False,
        'optimizer_name': 'adam',
        'learning_rate': 1e-4,
        'momentum': 0.0,
        'weight_decay': 1e-4,
        'epochs': 5,
    },
    {
        'name': 'mobilenetv2_pretrained',
        'model_type': 'mobilenet_v2',
        'pretrained': True,
        'optimizer_name': 'adam',
        'learning_rate': 1e-4,
        'momentum': 0.0,
        'weight_decay': 1e-5,
        'epochs': 5,
    },
    {
        'name': 'mobilenetv2_scratch',
        'model_type': 'mobilenet_v2',
        'pretrained': False,
        'optimizer_name': 'sgd',
        'learning_rate': 1e-3,
        'momentum': 0.9,
        'weight_decay': 1e-4,
        'epochs': 5,
    },
    {
        'name': 'inceptionv3_pretrained',
        'model_type': 'inception_v3',
        'pretrained': True,
        'optimizer_name': 'adam',
        'learning_rate': 1e-4,
        'momentum': 0.0,
        'weight_decay': 1e-5,
        'epochs': 5,
    },
    {
        'name': 'inceptionv3_scratch',
        'model_type': 'inception_v3',
        'pretrained': False,
        'optimizer_name': 'sgd',
        'learning_rate': 1e-3,
        'momentum': 0.9,
        'weight_decay': 1e-4,
        'epochs': 5,
    },
]

results = []

for index, experiment in enumerate(experiments, start=1):
    print('\\n' + '=' * 80)
    print(f\"Experiment {index}/{len(experiments)}: {experiment['name']}\")
    print('=' * 80)

    history = trainer.train_model(
        model_type=experiment['model_type'],
        pretrained=experiment['pretrained'],
        device=cfg.DEVICE,
        epochs=experiment['epochs'],
        learning_rate=experiment['learning_rate'],
        optimizer_name=experiment['optimizer_name'],
        momentum=experiment['momentum'],
        weight_decay=experiment['weight_decay'],
    )

    results.append({
        'name': experiment['name'],
        'model_type': experiment['model_type'],
        'pretrained': experiment['pretrained'],
        'optimizer_name': experiment['optimizer_name'],
        'learning_rate': experiment['learning_rate'],
        'epochs': experiment['epochs'],
        'best_val_accuracy': history['best_val_accuracy'],
        'best_epoch': history['best_epoch'],
        'test_accuracy': history.get('test_accuracy'),
        'test_loss': history.get('test_loss'),
    })

results_dir = REPO_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
results_path = results_dir / 'all_experiments.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print('\\nAll experiments finished.')
print(f'Results saved to: {results_path}')

## Results Summary

The next cell ranks the runs by held-out test accuracy and lists the checkpoint files produced by the training pipeline.

In [ ]:
sorted_results = sorted(results, key=lambda item: item['test_accuracy'] if item['test_accuracy'] is not None else -1.0, reverse=True)

print('Top runs by test accuracy:')
for entry in sorted_results:
    print(f\"- {entry['name']}: test_accuracy={entry['test_accuracy']:.4f} | best_val_accuracy={entry['best_val_accuracy']:.4f} | best_epoch={entry['best_epoch']}\")

checkpoint_dir = REPO_ROOT / 'checkpoints'
if checkpoint_dir.exists():
    checkpoint_files = sorted(checkpoint_dir.glob('*.pth'))
    print('\\nCheckpoint files:')
    for checkpoint_file in checkpoint_files:
        print(f'- {checkpoint_file.name}')
else:
    print('No checkpoint directory was created.')

## Output Artifacts

The training history for every run is written to `results/all_experiments.json` in the notebook workspace.

In [ ]:
results_dir = REPO_ROOT / 'results'
results_path = results_dir / 'all_experiments.json'

print(f'Results file exists: {results_path.exists()}')
if results_path.exists():
    print(f'Results path: {results_path}')
    print(results_path.read_text()[:1000])
else:
    print('No results file found yet. Run the training cell first.')